Tests will be done with different models, including a HuggingFaceModel and an Ollama model, to determine response time. This will help us improve the results in the output model.

El presente proyecto lo dividiremos en dos partes, correspondientes a los dos modelos a implementar: uno de ellos será el modelo de contexto y el otro, el modelo de clasificación. El primero recibirá una imagen y buscará realizar su clasificación por medio de un modelo ya preentrenado, el segundo será un modelo local, configurado en el entorno local del tester.

## Estandar

1. Codigo se desarrollara en ingles
2. Los prompts en español
3. Usar visualizadores

# Test Time Models

#### Conection with Hugging Face

### First part

In [15]:
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import ChatHuggingFace
from langchain_community.document_loaders import ImageCaptionLoader
from langchain.chat_models import init_chat_model
from transformers import pipeline
from transformers import BlipProcessor, BlipForConditionalGeneration
from datasets import load_dataset


List of models to test
- (Toca buscar en la biblioteca de Hugging)


**Inference**
This models will help us to make the inference we are using them to produce new outputs
on new inputs taking advantage of the knowledge of the training of the model 
  


In [16]:
from PIL import Image
import httpx
from io import BytesIO
from transformers import AutoProcessor, BlipForConditionalGeneration



In [17]:
#With lang
# We will load the model and the ApiToken ,to set the parameters 
import os
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


In [ ]:
pil_image = Image.open("Testmodels/utils/Paisaje.jpg").convert("RGB")#este es 64  con el id y otros atributos 
inputs = processor(images=pil_image, return_tensors="pt")
out = model.generate(**inputs)
result1 = processor.decode(out[0], skip_special_tokens=True)

In [4]:
print(result1)

a lone tree in a field of green grass


## Chains

List of models to test
- gemma:2b
- gemma:7b
- mistral:7b
- mixtral:8x7b
- llama3:7b
- llama3:70b
- phi3:mini
- phi3:medium
- llava:7b
- llava:3.1

## Second Part

In [7]:
import langchain
print(langchain.__version__)


1.2.12


In [8]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

In [9]:
from langchain_core.runnables import RunnableConfig
from typing_extensions import Annotated, TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langgraph.runtime import Runtime
from langgraph.graph import StateGraph, add_messages 

In [10]:

llm = ChatOllama(
    model="gemma:2b", #Here you will need to update the model name to the one you have in ollama  check it with ollama list command 
    temperature=0,
)

### if we try it with the old version of lang is with chains  dont execute

In [ ]:
# we will work with 2 chains 1 for TTS and another to the TTA
template1 = (
    "You are a helpful assistant. "
    "Tu rol es generar respuestas claras, rápidas y útiles, adaptadas al contexto del usuario. "
    "Debes mantener un tono profesional pero cercano, y producir descripciones concisas cuando se trate de imágenes. "
    "Siempre prioriza la precisión y la brevedad."
)

prompt1 = ChatPromptTemplate.from_messages([
    ("system", template1),
    ("human",
     "Genera una descripción breve y clara de la imagen basada en el mensaje {result}. "
     "La descripción debe ser rápida, concisa y reflejar el entorno de la imagen."
    )
])

chain1 = LLMChain(llm=llm, prompt=prompt1, output_key="description1")

#here if we want to run one by one
#fist_action = chain1.invoke({"result": result})
#print(first_action["description1"])


In [ ]:

template2 = (
    "You are a helpful assistant. "
    "Tu rol es generar respuestas claras, rápidas y útiles, adaptadas al contexto del usuario. "
    "Debes mantener un tono profesional pero cercano, y producir descripciones concisas cuando se trate de imágenes. "
    "Siempre prioriza la precisión y la brevedad."
)

prompt2 = ChatPromptTemplate.from_messages([
    ("system", template2),
    ("human",
     "Genera una descripción breve y clara de la imagen basada en el mensaje {result}. "
     "Basate en este listado de acciones que el robot puede realizar:\n\n"
     "Brazos:\n- Levantar brazo\n- Bajar brazo\n- Extender hacia adelante\n- Retractar hacia atrás\n- Abrir/cerrar mano\n\n"
     "Cabeza:\n- Girar a la izquierda\n- Girar a la derecha\n"
     "Parpadeo:\n- Cerrar ojos\n- Abrir ojos\n- Parpadeo rápido \n\n"
     "Torso:\n- Rotar ligeramente a izquierda/derecha\n\n"
     "Genera el resultado de acción."
    )
])

chain2 = LLMChain(llm=llm, prompt=prompt2, output_key="description2")

#here if we want to run one by one
#second_action = chain2.invoke({"result": result})
#print(ai_msg["description2"])


In [ ]:
#Here we will run the 2 chains in a sequence, the output of the first one will 
# be the input of the second one but we will get  both outputs
seq_chain = SimpleSequentialChain(chains=[chain1, chain2],
                                  input_variables=["result"], 
                                  output_variables=["description1", "description2"],
                                  verbose=True)

In [ ]:
print(f"Result of the first chain: {chain1.invoke({'result': result})}")


In [ ]:
print(f"Resul of the second chain: {chain2.invoke({'result': result})}")

### If we try with the new version is with nodes

In [11]:
from langchain_core.messages import AIMessage, HumanMessage

In [12]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

class Context(TypedDict):
    r: float    


#here we will have the firt node 
def modeltts(state: State) -> dict:
    template1 = (
        "You are a helpful assistant. "
        "Tu rol es generar respuestas claras, rápidas y útiles, adaptadas al contexto del usuario. "
        "Debes mantener un tono profesional pero cercano, y producir descripciones concisas. "
        "Siempre prioriza la precisión y la brevedad."
    )
    prompt1 = ChatPromptTemplate.from_messages([
        ("system", template1),
        ("human",
         "Genera una descripción breve y clara basada en el caption {result}. "
         "La descripción debe ser rápida, concisa y reflejar el entorno."
        )
    ])
    caption_text = state["messages"][-1].content
    response = llm.invoke(prompt1.format(result=caption_text))
    return {"messages": [AIMessage(content=response.content)]}



#here we will have the second node
def modeltta(state: State) -> dict:
    template2 = (
        "You are a helpful assistant. "
        "Tu rol es generar respuestas claras, rápidas y útiles, adaptadas al contexto del usuario. "
        "Debes mantener un tono profesional pero cercano, y producir descripciones concisas. "
        "Siempre prioriza la precisión y la brevedad."
    )
    prompt2 = ChatPromptTemplate.from_messages([
        ("system", template2),
        ("human",
         "Genera una acción basada en el caption '{result}'. "
         "Acciones posibles:\n"
         "- Levantar brazo\n- Bajar brazo\n- Extender hacia adelante\n- Retractar hacia atrás\n"
         "- Abrir/cerrar mano\n- Girar cabeza\n- Parpadeo\n- Rotar torso"
        )
    ])
    result_text = state["messages"][-1].content
    response = llm.invoke(prompt2.format(result=result_text))
    return {"messages": [AIMessage(content=response.content)]}



#here we will create the graph and add the nodes and edges, we will set the entry point and the finish point
graph = StateGraph(state_schema=State, context_schema=Context)
graph.add_node("modeltts", modeltts)
graph.add_node("modeltta", modeltta)
graph.add_edge("modeltts", "modeltta")
graph.set_entry_point("modeltts")
graph.set_finish_point("modeltta")


#here we will compile the graph to be able to run it
compiled = graph.compile()

In [14]:

#here we will execute the graph with the initial input and context, the output of the first node will be the input of the second node
result2 = compiled.invoke(
    {"messages": [HumanMessage(content=result1)]},
    context={"r": 3.0}
)
print(result2)


{'messages': [HumanMessage(content='a lone tree in a field of green grass', additional_kwargs={}, response_metadata={}, id='7a9057c2-1cd4-410a-ab0a-cd17b055bdbf'), AIMessage(content="The lone tree stands tall and proud in the vibrant green field of grass. Its branches sway gently in the gentle breeze, creating a sense of tranquility and serenity. The tree's silhouette is a stark contrast against the lush greenery, highlighting its resilience and presence amidst the vibrant surroundings.", additional_kwargs={}, response_metadata={}, id='e552a3c0-60b7-4a17-8738-51bfca4b3d02', tool_calls=[], invalid_tool_calls=[]), AIMessage(content="The lone tree stands tall and proud in the vibrant green field of grass. Its branches sway gently in the gentle breeze, creating a sense of tranquility and serenity. The tree's silhouette is a stark contrast against the lush greenery, highlighting its resilience and presence amidst the vibrant surroundings.", additional_kwargs={}, response_metadata={}, id='8a